# Blender render on Kaggle — background, frames pushed to Google Drive

Renders a Blender project on Kaggle's free T4 and uploads **each frame to Google Drive
as it finishes**.

**Why Kaggle instead of Colab:** you can close the tab. *Save Version → Save & Run All
(Commit)* runs the whole notebook server-side for up to **12 hours**. The Colab version
of this workflow needed the tab open and died after roughly 90 minutes.

| | |
|---|---|
| Session limit | 12 h (GPU) |
| Weekly GPU quota | 30 h per the settings page. See the caveat below — the API disagrees. |
| `/kaggle/working` | 20 GB, persisted |
| `/kaggle/tmp` | ~60 GB, **not** persisted — renders go here, then upload |

**Frames upload one at a time, as each completes.** If the wall hits or the session
dies, every finished frame is already safe on Drive. Re-running skips what is already
there, so a long animation can be finished across several commits.

> **Two sources disagree about the GPU quota — check before planning a long render.**
> On 2026-07-30 the Kaggle settings page showed **30 hrs** while the API's
> `gpuQuota.totalTimeAllowed` returned **21600 s (6 h)**, for the same account at the
> same time. TPU agreed across both (72000 s = 20 hrs), so the API is reading real
> fields — but which number is *enforced* for GPU is unresolved. `MAX_RUNTIME_HOURS`
> below is set to 5.0, which is safe under either reading. Raise it only after you have
> watched a real run and know which figure binds. Check the API side with:
> ```python
> from kagglesdk import KaggleClient
> from kagglesdk.kernels.types.kernels_api_service import ApiGetAcceleratorQuotaStatisticsRequest
> print(KaggleClient().kernels.kernels_api_client.get_accelerator_quota_statistics(
>     ApiGetAcceleratorQuotaStatisticsRequest()))
> ```

## Before your first run

1. **Settings → Accelerator → GPU T4** (`P100` also works)
2. **Settings → Internet → On** — requires a phone-verified Kaggle account. Without it
   nothing downloads and rclone cannot reach Drive.
3. **Add-ons → Secrets** — add a secret named `RCLONE_CONF` (next cell explains how)

## One-time setup: the `RCLONE_CONF` secret

Kaggle has no `drive.mount`. Access goes through rclone with **your own OAuth token**.

> **Why not a service account?** Service accounts created after **15 April 2025 can no
> longer own Google Drive items**, so the old "share a folder with a service account"
> recipe no longer works for a personal Gmail Drive — it only works on a Workspace
> Shared Drive. A user OAuth token is the working path, and it keeps the rendered files
> owned by you, under your normal storage quota.

**On your own machine** (not here) — install rclone, then:

```bash
rclone config
#  n) New remote
#  name> gdrive
#  Storage> drive
#  client_id>      <- paste your own (see note below)
#  client_secret>  <- paste your own
#  scope> 1        (full access)
#  Edit advanced config? n
#  Use web browser to automatically authenticate? y   <- opens a browser, sign in
#  Configure this as a Shared Drive? n
```

> Create your own `client_id`/`client_secret` at
> <https://console.cloud.google.com/apis/credentials> (enable the Drive API, then make an
> OAuth client ID of type *Desktop app*). rclone's built-in shared client_id is being
> retired during 2026, so leaving these blank will stop working.

Then print the config and copy **the whole thing**:

```bash
rclone config show gdrive
```

You get something like:

```ini
[gdrive]
type = drive
client_id = xxxx.apps.googleusercontent.com
client_secret = xxxx
scope = drive
token = {"access_token":"...","refresh_token":"...","expiry":"..."}
team_drive =
```

**In this notebook:** *Add-ons → Secrets → Add a new secret*
- Label: `RCLONE_CONF`
- Value: the entire block above, `[gdrive]` line included

Treat it like a password — it grants access to your Drive. Anyone you share this
notebook with does **not** get your secret, but do keep the notebook **private** while
you are iterating.

In [ ]:
# ============================== CONFIG ==============================
# Everything you normally change lives here.

# --- Blender -------------------------------------------------------
# ONE variable. The download URL, tarball name, extracted folder and
# binary path are all derived from it -- which is exactly what the old
# Colab notebook got wrong (it spelled the version three ways and the
# copy step failed).
BLENDER_VERSION = "5.2.0"        # latest stable LTS, released 2026-07-14
# Scenes authored in 4.2 may render differently under 5.x (shaders,
# geometry nodes, colour management). If output looks wrong, change the
# line above to "4.2.13" -- nothing else needs to change.

# --- Where things live on Drive ------------------------------------
DRIVE_REMOTE   = "gdrive"                  # the remote name in RCLONE_CONF
PROJECT_FOLDER = "malia"                   # folder under Blender/ on your Drive
BLEND_FILE     = "malia"                   # .blend name, WITHOUT the extension
OUTPUT_NAME    = "malia"                   # prefix for rendered frames

DRIVE_PROJECT_DIR = f"{DRIVE_REMOTE}:Blender/{PROJECT_FOLDER}"
DRIVE_OUTPUT_DIR  = f"{DRIVE_REMOTE}:Blender/render_results/{PROJECT_FOLDER}"

# --- What to render ------------------------------------------------
START_FRAME  = 1
END_FRAME    = 1                 # same as START_FRAME renders a single image
RESOLUTION_X = 1920
RESOLUTION_Y = 1920
SAMPLES      = 512
FPS          = 24
FILE_FORMAT  = "PNG"             # "PNG" or "JPEG"

# --- Run behaviour -------------------------------------------------
MAX_RUNTIME_HOURS = 5.0          # your GPU quota is 6h/WEEK (verified 2026-07-30),
                                 # not 30h. One run can eat all of it. Check with
                                 # kagglesdk GetAcceleratorQuotaStatistics.
SKIP_EXISTING     = True         # resume: don't re-render frames already on Drive
DEVICE            = "OPTIX"      # OPTIX is faster on T4 (RT cores); falls back to CUDA

# --- Derived (do not edit) -----------------------------------------
SERIES       = ".".join(BLENDER_VERSION.split(".")[:2])
TARBALL      = f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL  = f"https://download.blender.org/release/Blender{SERIES}/{TARBALL}"
BLENDER_DIR  = f"/kaggle/tmp/blender-{BLENDER_VERSION}-linux-x64"
BLENDER_BIN  = f"{BLENDER_DIR}/blender"
WORK_DIR     = "/kaggle/tmp/work"
PROJECT_DIR  = f"{WORK_DIR}/project"
RENDER_DIR   = f"{WORK_DIR}/frames"

import os, subprocess
for d in (WORK_DIR, PROJECT_DIR, RENDER_DIR):
    os.makedirs(d, exist_ok=True)

def sh(cmd, check=True):
    """Run a shell command and echo its output. Defined here so every later
    cell can use it even if you re-run cells out of order."""
    p = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.stdout:
        print(p.stdout.rstrip())
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

print(f"Blender  : {BLENDER_VERSION}")
print(f"Source   : {DRIVE_PROJECT_DIR}/{BLEND_FILE}.blend")
print(f"Dest     : {DRIVE_OUTPUT_DIR}/")
print(f"Frames   : {START_FRAME}..{END_FRAME}  ({END_FRAME - START_FRAME + 1} total)")
print(f"Settings : {RESOLUTION_X}x{RESOLUTION_Y}, {SAMPLES} samples, {FILE_FORMAT}")

In [ ]:
# ===================== MACHINE CHECK ================================
# What you actually get, and what will run out first.

import psutil, subprocess

vm    = psutil.virtual_memory()
cores = psutil.cpu_count(logical=True)
print(f"CPU        : {cores} cores")
print(f"System RAM : {vm.total/2**30:5.1f} GB total, {vm.available/2**30:5.1f} GB free")

gpus = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader",
    shell=True, capture_output=True, text=True).stdout.strip()

if not gpus:
    print("\n*** NO GPU. Settings -> Accelerator -> GPU T4 x2, or Cycles will")
    print("*** fall back to CPU and a frame that takes 2 min will take hours.")
else:
    print("\nGPUs:")
    for line in gpus.splitlines():
        print(f"  {line}")
    n = len(gpus.splitlines())
    if n > 1:
        print(f"\n{n} GPUs -> Cycles splits each frame across both, roughly {n}x faster.")
        print("Note this is NOT more usable VRAM: the whole scene must fit on EACH")
        print("card independently. Two 15 GB T4s render faster, they do not give 30 GB.")

print(f"""
Which limit bites first, for Cycles:
  System RAM {vm.total/2**30:.0f} GB  - loading the .blend, textures, Blender itself.
                    Generous; rarely the wall.
  VRAM       ~15 GB/GPU - THE REAL CEILING. Scene must fit or Cycles
                    spills to host memory and crawls.
  Disk       /kaggle/tmp ~60 GB scratch (not persisted)
             /kaggle/working 20 GB (persisted)

Per-frame RAM and VRAM are reported during the render below, so you can
see whether there is headroom to raise SAMPLES or RESOLUTION.
""")

In [ ]:
# ======================= INSTALL BLENDER ============================
import os, time

if os.path.exists(BLENDER_BIN):
    print("Blender already extracted, skipping download.")
else:
    t0 = time.time()
    print(f"Downloading {BLENDER_URL}")
    sh(f"wget -q --show-progress -O /kaggle/tmp/{TARBALL} '{BLENDER_URL}'")
    print("Extracting...")
    sh(f"tar -xf /kaggle/tmp/{TARBALL} -C /kaggle/tmp")
    os.remove(f"/kaggle/tmp/{TARBALL}")          # reclaim ~350 MB
    print(f"Done in {time.time()-t0:.0f}s")

# Sanity check: the binary must exist and report the version we asked for.
assert os.path.exists(BLENDER_BIN), (
    f"{BLENDER_BIN} not found. The extracted folder name did not match "
    f"BLENDER_VERSION={BLENDER_VERSION!r}. Check that this version exists at "
    f"https://download.blender.org/release/Blender{SERIES}/"
)
sh(f"{BLENDER_BIN} --version | head -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

In [ ]:
# ================= INSTALL + CONFIGURE RCLONE =======================
import os, subprocess

# --- install rclone (static binary, no apt) ---
if subprocess.run("which rclone", shell=True, capture_output=True).returncode != 0:
    sh("wget -q -O /tmp/rclone.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip")
    sh("unzip -q -o /tmp/rclone.zip -d /tmp/rclone-dl")
    sh("cp /tmp/rclone-dl/rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone")
sh("rclone version | head -1")

# --- write the config from the Kaggle Secret ---
try:
    from kaggle_secrets import UserSecretsClient
    conf = UserSecretsClient().get_secret("RCLONE_CONF")
except Exception as e:
    raise SystemExit(
        "Could not read the RCLONE_CONF secret.\n"
        "Add-ons -> Secrets -> Add a new secret, label it exactly RCLONE_CONF, and\n"
        "paste the output of `rclone config show gdrive` from your own machine.\n"
        f"Underlying error: {e}"
    )

if f"[{DRIVE_REMOTE}]" not in conf:
    raise SystemExit(
        f"RCLONE_CONF does not define a remote called [{DRIVE_REMOTE}].\n"
        f"Either rename the remote in the secret, or change DRIVE_REMOTE in CONFIG."
    )

cfg_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(cfg_dir, exist_ok=True)
with open(os.path.join(cfg_dir, "rclone.conf"), "w") as f:
    f.write(conf.strip() + "\n")
os.chmod(os.path.join(cfg_dir, "rclone.conf"), 0o600)

# --- prove Drive is actually reachable BEFORE spending GPU time ---
rc = sh(f"rclone lsd {DRIVE_REMOTE}: --max-depth 1", check=False)
if rc != 0:
    raise SystemExit(
        "rclone could not reach Drive. Common causes:\n"
        "  - Settings -> Internet is Off (needs a phone-verified account)\n"
        "  - the token in RCLONE_CONF has been revoked; re-run `rclone config` locally\n"
        "  - client_id/client_secret missing (rclone's shared ones are being retired)"
    )
print("\nDrive reachable.")

In [ ]:
# ================= PULL THE PROJECT FROM DRIVE ======================
# Copies the whole project folder so linked textures / caches come too.

print(f"Syncing {DRIVE_PROJECT_DIR} -> {PROJECT_DIR}")
sh(f"rclone copy '{DRIVE_PROJECT_DIR}' '{PROJECT_DIR}' --transfers 8 --stats 5s --stats-one-line")

blend_path = f"{PROJECT_DIR}/{BLEND_FILE}.blend"
if not os.path.exists(blend_path):
    found = []
    for root, _, files in os.walk(PROJECT_DIR):
        found += [os.path.join(root, f) for f in files if f.endswith(".blend")]
    raise SystemExit(
        f"{BLEND_FILE}.blend not found in {DRIVE_PROJECT_DIR}.\n"
        f".blend files that ARE there: {found or 'none'}\n"
        f"Remember BLEND_FILE must NOT include the .blend extension."
    )

size_mb = os.path.getsize(blend_path) / 1e6
print(f"\nFound {blend_path} ({size_mb:.1f} MB)")

In [ ]:
%%writefile /kaggle/working/render_setup.py
# Runs INSIDE Blender (blender -b file.blend -P render_setup.py -f N).
# Reads settings from environment variables so there is no fragile quoting
# in the shell command -- the old notebook built five --python-expr strings
# with nested escaped quotes, which is where it became unreadable.
import os, bpy

scene = bpy.context.scene

scene.render.resolution_x = int(os.environ["BR_RES_X"])
scene.render.resolution_y = int(os.environ["BR_RES_Y"])
scene.render.resolution_percentage = 100
scene.render.fps = int(os.environ["BR_FPS"])
scene.render.image_settings.file_format = os.environ["BR_FORMAT"]
scene.render.filepath = os.environ["BR_OUTPUT"]
scene.render.engine = "CYCLES"
scene.cycles.samples = int(os.environ["BR_SAMPLES"])

# --- enable the GPU ---
# Setting cycles.device = "GPU" alone is not enough: the devices themselves
# must be switched on in preferences, or Blender silently renders on CPU.
prefs = bpy.context.preferences.addons["cycles"].preferences
wanted = os.environ.get("BR_DEVICE", "OPTIX")

chosen = None
for backend in (wanted, "CUDA", "NONE"):
    if backend == "NONE":
        break
    try:
        prefs.compute_device_type = backend
        prefs.refresh_devices()
        if any(d.type == backend for d in prefs.devices):
            chosen = backend
            break
    except TypeError:
        continue          # this build does not support that backend

if chosen:
    # Only enable devices belonging to the CHOSEN backend. Using
    # `d.type != "CPU"` enabled the same physical GPU twice (it appears
    # once per backend) -- observed on a Kaggle P100 on 2026-07-30.
    for d in prefs.devices:
        d.use = (d.type == chosen)
    scene.cycles.device = "GPU"
    active = [d.name for d in prefs.devices if d.use]
    print(f"[setup] rendering on {chosen}: {active}")
else:
    scene.cycles.device = "CPU"
    print("[setup] WARNING: no GPU backend available, falling back to CPU "
          "(this will be very slow -- check Settings -> Accelerator)")

print(f"[setup] {scene.render.resolution_x}x{scene.render.resolution_y} "
      f"{scene.cycles.samples} samples -> {scene.render.filepath}")

In [ ]:
# ========================== RENDER LOOP =============================
# One frame per Blender invocation, uploaded immediately, deleted locally.
#
# Why per-frame rather than Blender's own -a animation flag:
#   - a frame lands on Drive the moment it is done, so a 12h timeout or a
#     dead session costs you at most one frame
#   - re-running skips finished frames, so a long animation can be split
#     across several commits
#   - local disk stays flat instead of accumulating the whole sequence

import os, re, time, glob, subprocess, psutil

ext = ".png" if FILE_FORMAT == "PNG" else ".jpg"

def mem_report():
    """Peak-ish RAM + VRAM sampled right after a frame finishes."""
    vm = psutil.virtual_memory()
    ram = f"RAM {vm.used/2**30:.1f}/{vm.total/2**30:.0f}GB"
    out = subprocess.run(
        "nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader,nounits",
        shell=True, capture_output=True, text=True).stdout.strip()
    vram = []
    for i, line in enumerate(out.splitlines()):
        try:
            used, total = [int(x) for x in line.split(",")]
            vram.append(f"GPU{i} {used/1024:.1f}/{total/1024:.0f}GB")
        except ValueError:
            pass
    return ram + ("  " + "  ".join(vram) if vram else "")

# --- which frames are already on Drive? ---
done = set()
if SKIP_EXISTING:
    p = subprocess.run(f"rclone lsf '{DRIVE_OUTPUT_DIR}'", shell=True,
                       capture_output=True, text=True)
    for name in p.stdout.splitlines():
        m = re.search(rf"{re.escape(OUTPUT_NAME)}_(\d+){re.escape(ext)}$", name)
        if m:
            done.add(int(m.group(1)))
    if done:
        print(f"{len(done)} frame(s) already on Drive, will be skipped.\n")

todo = [f for f in range(START_FRAME, END_FRAME + 1) if f not in done]
if not todo:
    print("Nothing to do -- every requested frame is already on Drive.")
else:
    print(f"Rendering {len(todo)} frame(s): {todo[0]}..{todo[-1]}\n")

env = os.environ.copy()
env.update({
    "BR_RES_X":   str(RESOLUTION_X),
    "BR_RES_Y":   str(RESOLUTION_Y),
    "BR_FPS":     str(FPS),
    "BR_SAMPLES": str(SAMPLES),
    "BR_FORMAT":  FILE_FORMAT,
    "BR_DEVICE":  DEVICE,
    "BR_OUTPUT":  f"{RENDER_DIR}/{OUTPUT_NAME}_",   # Blender appends ####
})

t_start = time.time()
rendered, failed = [], []

for frame in todo:
    elapsed_h = (time.time() - t_start) / 3600
    if elapsed_h > MAX_RUNTIME_HOURS:
        print(f"\nReached MAX_RUNTIME_HOURS ({MAX_RUNTIME_HOURS}h). "
              f"Stopping cleanly with {len(rendered)} frame(s) uploaded.")
        print("Re-run this notebook to continue -- finished frames will be skipped.")
        break

    t0 = time.time()
    print(f"--- frame {frame} ---", flush=True)
    p = subprocess.run(
        [BLENDER_BIN, blend_path, "-b", "-noaudio",
         "-P", "/kaggle/working/render_setup.py", "-f", str(frame)],
        env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    # Blender is extremely chatty; show the tail only, plus anything alarming.
    tail = p.stdout.strip().splitlines()[-3:]
    print("\n".join(tail))

    produced = glob.glob(f"{RENDER_DIR}/{OUTPUT_NAME}_*{ext}")
    if p.returncode != 0 or not produced:
        print(f"!! frame {frame} FAILED (exit {p.returncode}) -- full log:")
        print(p.stdout[-3000:])
        failed.append(frame)
        continue

    local = produced[0]
    sh(f"rclone copy '{local}' '{DRIVE_OUTPUT_DIR}' --stats-one-line", check=False)
    os.remove(local)                       # keep /kaggle/tmp flat
    rendered.append(frame)
    print(f"    {time.time()-t0:.0f}s  {mem_report()}  "
          f"({len(rendered)}/{len(todo)} done)\n", flush=True)

print("=" * 60)
print(f"Rendered  : {len(rendered)}  {rendered if rendered else ''}")
print(f"Failed    : {len(failed)}  {failed if failed else ''}")
print(f"Elapsed   : {(time.time()-t_start)/60:.1f} min")
print(f"Output    : {DRIVE_OUTPUT_DIR}/")

## Running it in the background

Once a single frame renders correctly interactively:

1. **Save Version** (top right)
2. Choose **Save & Run All (Commit)**
3. Close the tab

It keeps running server-side for up to 12 hours. Frames appear in
`Blender/render_results/<project>/` on your Drive as they complete — you can watch the
folder fill up from your phone.

**For an animation longer than one session:** set `START_FRAME`/`END_FRAME` to the full
range and just commit again when it stops. `SKIP_EXISTING` means each run picks up where
the last left off.

**Watch your quota.** ~30 GPU-hours per week. A committed run consumes quota for as long
as it runs, so cancel runs you no longer want under *Notebook → Active Events*.

### If something goes wrong

| Symptom | Cause |
|---|---|
| `Could not read the RCLONE_CONF secret` | Secret missing or misnamed. Must be exactly `RCLONE_CONF`. |
| `rclone could not reach Drive` | Internet off (needs phone verification), or the OAuth token was revoked. |
| `<name>.blend not found` | `BLEND_FILE` must omit the `.blend` extension. The error lists what it did find. |
| `no GPU backend available, falling back to CPU` | Accelerator is not set to GPU. Stop and fix — CPU Cycles is unusably slow here. |
| Render looks wrong vs. Blender 4.2 | The 4.2 → 5.2 jump. Set `BLENDER_VERSION = "4.2.13"` and re-run. |